# **AML Challenge: Model Stitching**

## Import libraries

In [15]:
# Image libraries and visualization
import numpy as np
import pandas as pd
from eval_2 import evaluate_model_on_validation

# Torch and torchvision
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# Utilities
import time
import random

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


### **Translator Model:**

In [16]:
class TranslatorDINOv2(nn.Module):
    def __init__(self, text_dim=1024, image_dim=1536, hidden_dim=3072):
        super().__init__()
        # Temperature parameter for scaling logits. Initialized to log(1/0.07)
        self.logit_scale = nn.Parameter(torch.tensor(np.log(1 / 0.07)))

        # ENCODER: Two fully-connected blocks to transform the text embedding into a richer latent representation.
        # LayerNorm + GELU improve stability and non-linearity.

        self.encoder = nn.ModuleList([
            nn.Sequential(
                nn.Linear(text_dim, hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.GELU()
            ),
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim // 2),
                nn.LayerNorm(hidden_dim // 2),
                nn.GELU(),
                nn.Dropout(0.6)  # deactivated 60% neurons for regularization and to prevent overfitting
            ),
        ])

        # DECODER: Maps the compressed representation back to the target DINOv2 image embedding dimension.

        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim // 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, image_dim),
            nn.Dropout(0.1) # deactivated 10% neurons for regularization
        )

    def forward(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(0)
        h = x
        for block in self.encoder:
            h = block(h)
        return self.decoder(h)


# Final translator model used for submission
translator_dinov2 = TranslatorDINOv2(
    text_dim=1024,
    image_dim=1536,
    hidden_dim=3840).to(device)

translator_dinov2.eval()

total_params = sum(p.numel() for p in translator_dinov2.parameters())
trainable_params = sum(p.numel() for p in translator_dinov2.parameters() if p.requires_grad)

print(f"   Total parameters: {total_params:,}")
print(f"   trainable_parameters: {trainable_params:,}")


   Total parameters: 24,606,337
   trainable_parameters: 24,606,337


In [17]:
def train_model(
    model,
    train_loader,
    val_loader,
    device,
    num_epochs=20,
    lr=1e-4,
    warmup_epochs=3,
    weight_decay=0.01
):
    """
    Train a bidirectional semantic alignment model (text ↔ image embeddings)
    using a symmetric InfoNCE objective.

    The function normalizes the target embeddings, computes text→image and
    image→text contrastive losses, and optimizes the model with AdamW.
    A warm-up phase is applied to the learning rate, followed by cosine decay.

    The loss is computed symmetrically by averaging the text-to-image and
    image-to-text cross-entropy terms. Gradient clipping is applied at each step,
    and the learnable logit scaling factor is constrained to a stable numeric range.

    The function tracks training and validation loss across epochs, updates the
    scheduler, and finally saves the trained model to disk.
    """

    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    # Scheduler: warmup + cosine decay
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return float(epoch + 1) / float(max(1, warmup_epochs))
        progress = float(epoch - warmup_epochs) / float(max(1, num_epochs - warmup_epochs))
        return 0.5 * (1.0 + np.cos(np.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    history = {'train_loss': [], 'val_loss': [], 'lr': []}
    start_time_total = time.time()

    for epoch in range(num_epochs):
        model.train()
        total_train_loss = 0.0

        # ---------------------- TRAIN ----------------------
        for text_emb, target_img_emb in train_loader:
            text_emb = text_emb.to(device, non_blocking=True)
            target_img_emb = F.normalize(target_img_emb.to(device), dim=-1)

            optimizer.zero_grad(set_to_none=True)

            # Forward: text → image prediction
            pred_img_emb = model(text_emb)

            # Symmetric InfoNCE
            logit_scale = model.logit_scale.exp().clamp(1e-2, 100) # prevent extreme scaling
            logits_t2i = logit_scale * pred_img_emb @ target_img_emb.T
            logits_i2t = logit_scale * target_img_emb @ pred_img_emb.T
            labels = torch.arange(len(logits_t2i), device=device)

            loss_t2i = criterion(logits_t2i, labels)
            loss_i2t = criterion(logits_i2t, labels)
            loss = 0.5 * (loss_t2i + loss_i2t)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            # Keep logit_scale in a stable numerical range
            with torch.no_grad():
                model.logit_scale.clamp_(np.log(1/0.1), np.log(100))

            total_train_loss += loss.item()

        train_loss = total_train_loss / len(train_loader)

        # ---------------------- VALIDATION ----------------------
        # Evaluation mode for validation step
        model.eval()
        total_val_loss = 0.0
        with torch.no_grad():
            for text_emb, target_img_emb in val_loader:
                text_emb = text_emb.to(device, non_blocking=True)
                target_img_emb = F.normalize(target_img_emb.to(device), dim=-1)
                pred_img_emb = model(text_emb)

                logit_scale = model.logit_scale.exp().clamp(1e-2, 100)
                logits_t2i = logit_scale * pred_img_emb @ target_img_emb.T
                logits_i2t = logit_scale * target_img_emb @ pred_img_emb.T
                labels = torch.arange(len(logits_t2i), device=device)

                loss_t2i = criterion(logits_t2i, labels)
                loss_i2t = criterion(logits_i2t, labels)
                val_loss = 0.5 * (loss_t2i + loss_i2t)
                total_val_loss += val_loss.item()

        val_loss = total_val_loss / len(val_loader)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]
        history['lr'].append(current_lr)

        if epoch % 5 == 0 or epoch == num_epochs - 1:
            print(f"Epoch [{epoch+1}/{num_epochs}] | Train: {train_loss:.5f} | Val: {val_loss:.5f} | LR: {current_lr:.1e}")

    total_time = time.time() - start_time_total
    torch.save(model.state_dict(), "translator_dinov2_final.pth")
    print(f"Training completed in {total_time/60:.1f} min — model saved to 'translator_dinov2_final.pth'")

    model.eval()
    return model, history


## Import Train Data

## Split Train and Validation

In [18]:
train_data = np.load("train.npz")
caption_embd = train_data['captions/embeddings']
image_embd = train_data['images/embeddings']

# Map caption embeddings to corresponding image embeddings
label = train_data['captions/label'] # N x M

# repeat the image embeddings according to the label
label_idx = np.nonzero(label)[1]
image_embd = image_embd[label_idx]
assert caption_embd.shape[0] == image_embd.shape[0], "Mismatch in number of caption and image embeddings"

X = torch.from_numpy(caption_embd).float()

# Map each caption to its corresponding image embedding
y = torch.from_numpy(image_embd).float()
label = torch.from_numpy(label).bool()

print(f"Train data: {len(X)} captions, {len(np.unique(label_idx))} images")

Train data: 125000 captions, 25000 images


In [19]:
# 90% for training, 10% for validation

n_train = int(0.9 * len(X))
TRAIN_SPLIT = torch.zeros(len(X), dtype=torch.bool)
TRAIN_SPLIT[:n_train] = 1
X_train, X_val = X[TRAIN_SPLIT], X[~TRAIN_SPLIT]
y_train, y_val = y[TRAIN_SPLIT], y[~TRAIN_SPLIT]


train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

# dataloaders
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True) # increased batch size for stability and more negatives
val_loader = DataLoader(val_dataset, batch_size=512)
y_train.shape[-1], X_val.shape[-1]

(1536, 1024)

# Optuna grid search

To identify the optimal Hyperparameters, we adopted the best configurations for our Translator Model as determined through *Optuna* Library.

**Warning** : the runtime for this search is approximately 45–75 minutes for 25 trials. You can find the best configuration in the *Retrain* cell.

In [ ]:
# best_trial_mrr = 0.0  # initialize best MRR
# best_trial_model_state = None
# best_trial_results = None

# def objective(trial: optuna.Trial) -> float:
#     global best_trial_mrr, best_trial_model_state, best_trial_results

#     # Hyperparameters to optimize
#     lr = trial.suggest_float('lr', 5e-4, 5e-3, log=True)
#     warmup_epochs = trial.suggest_int('warmup_epochs', 1, 5)
#     num_epochs = trial.suggest_int('num_epochs', 10, 15)
#     hidden_dim = 3840  # fixed for this experiment

#     print(f"\nTrial {trial.number} | lr={lr:.2e}, warmup={warmup_epochs}, epochs={num_epochs}")

#     # Create model
#     model_trial = TranslatorDINOv2(
#         text_dim=1024,
#         image_dim=1536,
#         hidden_dim=hidden_dim
#     ).to(device)

#     # Training
#     start = time.time()
#     _ = train_model(
#         model=model_trial,
#         train_loader=train_loader,
#         val_loader=val_loader,
#         device=device,
#         num_epochs=num_epochs,
#         lr=lr,
#         warmup_epochs=warmup_epochs,
#         weight_decay=0.01
#     )
#     duration = time.time() - start

#     # Validation evaluation
#     results = evaluate_model_on_validation(model_trial, val_dataset, device)
#     final_mrr = results["mrr"]

#     print(f"Trial {trial.number} - MRR: {final_mrr:.4f} ({duration/60:.1f} min)")

#     # Save best result
#     if final_mrr > best_trial_mrr:
#         best_trial_mrr = final_mrr
#         best_trial_results = results.copy()
#         best_trial_model_state = copy.deepcopy(model_trial.state_dict())

#         torch.save(model_trial.state_dict(), "translator_dinov2_best_trial.pth")

#         print(f"Best updated : MRR = {final_mrr:.4f}")

#     return final_mrr


In [ ]:
# study_name = "translator_optimization_trial"

# study = optuna.create_study(
#         study_name=study_name,
#         direction='maximize',
#         load_if_exists=True,
#         pruner=optuna.pruners.MedianPruner(
#             n_startup_trials=5,
#             n_warmup_steps=3,
#             interval_steps=1
#         ),
#         sampler=optuna.samplers.TPESampler(
#             n_startup_trials=10,
#             multivariate=True,
#             seed=42
#         )
#     )

# n_trials = 25 # Number of trials to run
# study.optimize(objective,n_trials=n_trials,show_progress_bar=True,catch=(Exception,))

## Retraning on the full Dataset

In [20]:
seed = 42345 # Set a seed for reproducibility

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

X_full = torch.from_numpy(caption_embd).float()
y_full = torch.from_numpy(image_embd).float()

# use the full dataset for training
full_loader = DataLoader(TensorDataset(X_full, y_full), batch_size=512, shuffle=True)

# initialize TranslatorDINOv2 for full retraining
model_full = TranslatorDINOv2(1024, 1536, 3840).to(device) # initialize new model for full retraining

# AdamW optimizer with weight decay
optimizer = torch.optim.AdamW(model_full.parameters(), lr=0.0004, weight_decay=0.01)
criterion = nn.CrossEntropyLoss() # cross-entropy on similarity logits

num_epochs = 13
warmup_epochs = 3 # warm-up for first 3 epochs

def lr_lambda(epoch):
    # linear warmup for the first `warmup_epochs`
    if epoch < warmup_epochs:
        return epoch / float(max(1, warmup_epochs))
    # cosine decay for the remaining epochs
    progress = (epoch - warmup_epochs) / float(max(1, num_epochs - warmup_epochs))
    return 0.5 * (1.0 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# training loop on the full dataset
for epoch in range(num_epochs):
    model_full.train()
    total_loss = 0.0

    for text_emb, img_emb in full_loader:
        text_emb = text_emb.to(device)
        img_emb = img_emb.to(device)

        optimizer.zero_grad(set_to_none=True)

        pred = model_full(text_emb)
        # Normalize only image embeddings. The model learns to map text embeddings into this space
        img_emb = F.normalize(img_emb, dim=-1)

        logit_scale = model_full.logit_scale.exp()
        logits_t2i = (pred @ img_emb.T) * logit_scale
        logits_i2t = (img_emb @ pred.T) * logit_scale
        labels = torch.arange(len(logits_t2i), device=device)

        loss = (criterion(logits_t2i, labels) + criterion(logits_i2t, labels)) / 2 # symmetric InfoNCE loss
        loss.backward() # backpropagation

        torch.nn.utils.clip_grad_norm_(model_full.parameters(), 1.0)
        optimizer.step()

        # Clamp logit_scale for numerical stability
        with torch.no_grad():
            model_full.logit_scale.clamp_(np.log(1/0.1), np.log(100)) # avoid extreme values

        total_loss += loss.item()

    scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}] - "f"Full-data loss: {total_loss/len(full_loader):.3f} - "f"logit_scale={model_full.logit_scale.exp().item():.2f}")

torch.save(model_full.state_dict(), 'translator_dinov2_full_retrain.pth')
print("Final retraining completed. Model saved to 'translator_dinov2_full_retrain.pth'.")

results_retrain = evaluate_model_on_validation(model_full, val_dataset, device)
print(f"MRR after full retraining: {results_retrain['mrr']:.4f}")


Epoch [1/13] - Full-data loss: 14.350 - logit_scale=14.29
Epoch [2/13] - Full-data loss: 4.473 - logit_scale=13.89
Epoch [3/13] - Full-data loss: 2.182 - logit_scale=13.13
Epoch [4/13] - Full-data loss: 1.810 - logit_scale=11.93
Epoch [5/13] - Full-data loss: 1.453 - logit_scale=11.13
Epoch [6/13] - Full-data loss: 1.201 - logit_scale=10.65
Epoch [7/13] - Full-data loss: 1.005 - logit_scale=10.38
Epoch [8/13] - Full-data loss: 0.847 - logit_scale=10.24
Epoch [9/13] - Full-data loss: 0.695 - logit_scale=10.21
Epoch [10/13] - Full-data loss: 0.574 - logit_scale=10.22
Epoch [11/13] - Full-data loss: 0.481 - logit_scale=10.24
Epoch [12/13] - Full-data loss: 0.419 - logit_scale=10.26
Epoch [13/13] - Full-data loss: 0.385 - logit_scale=10.26
Final retraining completed. Model saved to 'translator_dinov2_full_retrain.pth'.
Generating predictions for 12,500 validation samples...
   Searching among 12,500 validation images
🎯 Results:
   MRR:        0.4539
   Recall@1:   0.1994
   Recall@5:   0.9

In [21]:
# define the submission model for inference

submission_model = TranslatorDINOv2(1024, 1536, 3840)
submission_model.load_state_dict(torch.load("translator_dinov2_full_retrain.pth", map_location=device))
submission_model.to(device)

C:\Users\flavi\AppData\Local\Temp\ipykernel_42732\1789167656.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  submission_model.load_state_dict(torch.load("translator_dino

TranslatorDINOv2(
  (encoder): ModuleList(
    (0): Sequential(
      (0): Linear(in_features=1024, out_features=3840, bias=True)
      (1): LayerNorm((3840,), eps=1e-05, elementwise_affine=True)
      (2): GELU(approximate='none')
    )
    (1): Sequential(
      (0): Linear(in_features=3840, out_features=1920, bias=True)
      (1): LayerNorm((1920,), eps=1e-05, elementwise_affine=True)
      (2): GELU(approximate='none')
      (3): Dropout(p=0.6, inplace=False)
    )
  )
  (decoder): Sequential(
    (0): Linear(in_features=1920, out_features=3840, bias=True)
    (1): LayerNorm((3840,), eps=1e-05, elementwise_affine=True)
    (2): ReLU()
    (3): Linear(in_features=3840, out_features=1536, bias=True)
    (4): Dropout(p=0.1, inplace=False)
  )
)

## Submission for Kaggle Competition

In [22]:
# import test data
test_data = np.load('test.clean.npz')
test_embds = test_data['captions/embeddings']
test_ids = test_data['captions/ids']

submission_model.eval()

test_embds_tensor = torch.from_numpy(test_embds).float()
with torch.no_grad():
    pred_embds = submission_model(test_embds_tensor.to(device)).cpu()
if isinstance(pred_embds, torch.Tensor):
    translated_embeddings = pred_embds.cpu().numpy()

# Save submission
submission_name = 'submission_standard_retrain_3840_GELU.csv'
df_submission = pd.DataFrame({'id': test_ids, 'embedding': translated_embeddings.tolist()})
df_submission.to_csv(submission_name, index=False)

print(f" Submission saved to '{submission_name}'")

 Submission saved to 'submission_standard_retrain_3840_GELU.csv'
